In [ ]:
from pathlib import Path

# Change these globals, then Run All. None selects the first source row.
DATASET = "fantastic-breaks"
SAMPLE_ID = None
CACHE_DIR = None
SAMPLING_SEED = 0
INITIALIZATION_SEED = 0

# Inspect assembly samples

Use `uv sync --group inspection` and select `.venv/bin/python` as the kernel.
This notebook calls the production adapters and preparation functions without
requiring an experiment or writing episodes. It supports every registered dataset.

Loading defaults to **non-streaming**: the first load prepares the entire `full`
split in the standard Hugging Face cache (or `CACHE_DIR`). Selecting one ID or
`limit=1` only bounds subsequent adaptation and geometry processing. Repeated loads
reuse the cache. Fantastic Breaks uses `writer_batch_size=1` for its high-resolution meshes.

All original mesh vertices and polygons are retained. Display triangulation is a
derivative, not a geometry repair or simplification. Mesh views may take time to
render. GT and source annotations stay in memory; clear outputs before saving/sharing.


In [ ]:
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/assembly_world_agent").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start this notebook inside the assembly-world-agent project.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from assembly_world_agent import PreparationConfig, load_samples, prepare_sample
from assembly_world_agent.utils import apply_pose, mesh_from_record, transform_points, triangulate

## 1. Load one pinned source sample

IDs remain strings, including leading zeros. The adapter pins a source revision;
no category, material, physical units or source split is inferred when absent.


In [ ]:
source = next(
    load_samples(
        DATASET,
        sample_ids=None if SAMPLE_ID is None else [SAMPLE_ID],
        limit=1,
        cache_dir=CACHE_DIR,
    )
)
display(
    pd.DataFrame(
        [
            {
                "dataset": source.dataset,
                "sample_id": source.sample_id,
                "revision": source.revision,
                "parts": len(source.parts),
                "source_splits": source.source_splits,
            }
        ]
    )
)
display(source.metadata)
display(
    pd.DataFrame(
        [
            {
                "part_id": p.part_id,
                "role": p.metadata.get("role"),
                "vertices": len(p.mesh.vertices),
                "polygons": len(p.mesh.faces),
                "normals": len(p.mesh.normals),
                "vertex_colors": len(p.metadata.get("vertex_colors", [])),
                "source_file": p.metadata.get("source_file"),
            }
            for p in source.parts
        ]
    )
)
display(
    {"annotation_keys": sorted(source.annotations), "source_equivalence": source.source_equivalence}
)

## 2. Inspect the supplied assembly frame

Drag to rotate, scroll to zoom, and click the legend to hide/show a part. Colors
follow part IDs throughout this notebook. Source and prepared coordinates have
different units; each view uses equal spatial aspect without rescaling individual parts.

For Fantastic Breaks the two parts are a scanned broken object and a synthetic
repair proxy. Identity axis conversion is a task convention: physical up is not
verified. The source annotation matrix is never applied.


In [ ]:
palette = px.colors.qualitative.Dark24
colors = {
    pid: palette[i % len(palette)] for i, pid in enumerate(sorted(p.part_id for p in source.parts))
}


def show_meshes(items, title):
    figure = go.Figure()
    for name, vertices, faces, color in items:
        figure.add_trace(
            go.Mesh3d(
                x=vertices[:, 0],
                y=vertices[:, 1],
                z=vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color=color,
                name=name,
                showlegend=True,
                lighting=dict(ambient=0.55, diffuse=0.8),
            )
        )
    figure.update_layout(
        title=title,
        height=650,
        scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
        legend=dict(itemclick="toggle", itemdoubleclick="toggleothers"),
    )
    figure.show()


show_meshes(
    (
        (
            p.part_id,
            apply_pose(p.mesh.vertices, p.assembled_pose),
            triangulate(p.mesh),
            colors[p.part_id],
        )
        for p in source.parts
    ),
    "Source assembly — supplied coordinates",
)

## 3. Prepare and verify the GT assembly

Use the same normalization, independent part frames, surface/FPS sampling and
initial placement as production. Map the reconstructed GT back to the source frame
to check that preprocessing preserves the assembly.


In [ ]:
sample = prepare_sample(
    source,
    PreparationConfig(
        sampling_seed=SAMPLING_SEED,
        initialization_seed=INITIALIZATION_SEED,
    ),
)
source_parts = {p.part_id: p for p in source.parts}
checks = []
for part in sample.parts:
    original = source_parts[part.part_id]
    expected = apply_pose(original.mesh.vertices, original.assembled_pose)
    restored = transform_points(
        apply_pose(part.mesh.vertices, part.gt_pose), sample.world_to_source
    )
    np.testing.assert_allclose(restored, expected, atol=1e-9, rtol=1e-9)
    checks.append(
        {
            "part_id": part.part_id,
            "max_source_reconstruction_error": float(np.max(np.abs(restored - expected))),
            "sampled_points": len(part.points),
        }
    )
display(pd.DataFrame(checks))
show_meshes(
    (
        (p.part_id, apply_pose(p.mesh.vertices, p.gt_pose), triangulate(p.mesh), colors[p.part_id])
        for p in sample.parts
    ),
    "Prepared GT assembly — normalized task coordinates",
)

## 4. Inspect the initial task layout

All parts move independently and are grounded and separated by the shared
preparation routine. This is a geometric initialization, not a physics stability test.


In [ ]:
show_meshes(
    (
        (
            p.part_id,
            apply_pose(p.mesh.vertices, p.initial_pose),
            triangulate(p.mesh),
            colors[p.part_id],
        )
        for p in sample.parts
    ),
    "Initial layout — normalized task coordinates",
)

## 5. Inspect production point clouds

These are the actual default 1000 FPS points per part, mapped by the same initial
and GT poses as the meshes. No independent visualization sampling is performed.


In [ ]:
figure = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["Initial point clouds", "GT point clouds"],
)
for column, pose_name in [(1, "initial_pose"), (2, "gt_pose")]:
    for part in sample.parts:
        points = apply_pose(part.points, getattr(part, pose_name))
        figure.add_trace(
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                marker=dict(size=2, color=colors[part.part_id]),
                name=part.part_id,
                legendgroup=part.part_id,
                showlegend=column == 1,
            ),
            row=1,
            col=column,
        )
figure.update_layout(
    height=600,
    title="Production surface/FPS samples",
    scene=dict(aspectmode="data"),
    scene2=dict(aspectmode="data"),
)
figure.show()
del figure

## 6. Fantastic Breaks reference and annotation inspection

The complete mesh below is **reference only, excluded from assembly inputs and GT**.
It is shown separately in its supplied coordinates and does not determine task
normalization. The annotation mask indexes broken-mesh vertices. Its matrix has
unverified direction and units and is displayed literally, never applied.
Other datasets skip this section.


In [ ]:
if source.dataset == "AssemblyWorld/fantastic-breaks":
    annotation = source.annotations["annotation"]
    mask = np.asarray(annotation["mask"], dtype=bool)
    display(
        pd.DataFrame(
            [
                {
                    "mask_dtype": annotation.get("mask_dtype"),
                    "mask_shape": annotation.get("mask_shape"),
                    "mask_mesh_role": annotation.get("mask_mesh_role"),
                    "mask_entries": mask.size,
                    "mask_true": int(mask.sum()),
                    "transform_dtype": annotation.get("transform_dtype"),
                    "transform_shape": annotation.get("transform_shape"),
                }
            ]
        )
    )
    display(pd.DataFrame(np.asarray(annotation["transform"])))
    reference = mesh_from_record(source.annotations["complete_reference"])
    show_meshes(
        [
            (
                "Complete reference (inspection only)",
                reference.vertices,
                triangulate(reference),
                "#999999",
            )
        ],
        "Complete reference — excluded from assembly inputs and GT",
    )
    del reference
else:
    print("No Fantastic Breaks reference inspection for this dataset.")

## Interpretation

Reconstruction checks establish preprocessing consistency, not model success.
The notebook does not execute an agent, score a prediction or simulate physics.
No private resource sidecars or episodes are written. Clear all outputs before
saving/sharing the notebook.
